# Chapter 15 — What Did the Model Actually See?

**Companion to *Applied AI*.**

This notebook accompanies Chapter 15. The chapter's finding is uncomfortable
and it is preserved on disk: **the record said the source paragraph had been
selected for the model, and the request that would have been sent did not
contain it.**

## Question

**Is "selected" the same as "sent"?**

## What this notebook establishes

- The five states of a piece of material, read from the preserved
  context-selection run: available, eligible, selected, rendered, received.
- A seal excluding sibling material **by declared lineage** — and admitting the
  same artifact once the lineage is removed.
- Two identity probes: a payload rewritten under the same id leaves the package
  identity **unchanged**, and an artifact id with no bytes behind it is
  selected anyway.
- The gap itself: the prepared request body, with the selected source absent
  from it.
- How Stage 15B closed that gap for calls that opt in — and what still sends
  the prompt string.

## What this notebook does **not** establish

- The preserved run used **one five-item fixture, four permutations, and no
  model**. Nothing here measures whether selection changes an answer.
- Declared sizes in the fixture are **synthetic estimates**, not tokenizer
  counts.
- The fifth state, *received*, is **not observable** from this side of the API,
  and no notebook can make it so.

## Setup

In [1]:
import json
import os
from pathlib import Path

def find_evidence_dir(marker="context-selection"):
    env = os.environ.get("APPLIED_AI_EVIDENCE")
    if env and Path(env).expanduser().is_dir():
        return Path(env).expanduser()
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        for cand in (base / "evidence",
                     base / "experiments" / "applied-ai" / "evidence"):
            if (cand / marker).is_dir():
                return cand
    raise FileNotFoundError("Set APPLIED_AI_EVIDENCE to the evidence directory.")

EVIDENCE_DIR = find_evidence_dir()
SEL = EVIDENCE_DIR / "context-selection" / "2026-09-13-54e32384"
REN = EVIDENCE_DIR / "context-rendering" / "2026-09-13-0f9a83b"

results = json.loads((SEL / "results.json").read_text(encoding="utf-8"))
fixture = json.loads((SEL / "fixture.json").read_text(encoding="utf-8"))

print("selection run   :", SEL.name)
print("network attempts:", results["network_attempts"])
print("model calls     :", results["model_calls"])

selection run   : 2026-09-13-54e32384
network attempts: 0
model calls     : 0


## 1. The five states

```text
available  !=  eligible  !=  selected  !=  rendered  !=  received
```

"Memory" is what is available. "Context" is what is selected for one operation.
Everything between them is a decision, and **exclusion is evidence**.

In [2]:
STATES = [
    ("available", "it exists: in the ledger, the artifact store, the claim record"),
    ("eligible",  "the operation is permitted to use it (no seal forbids it)"),
    ("selected",  "requirements and the budget admit it into the package"),
    ("rendered",  "its bytes were actually placed in the request"),
    ("received",  "the provider processed those bytes"),
]
for name, note in STATES:
    print(f"  {name:<12}{note}")

  available   it exists: in the ledger, the artifact store, the claim record
  eligible    the operation is permitted to use it (no seal forbids it)
  selected    requirements and the budget admit it into the package
  rendered    its bytes were actually placed in the request
  received    the provider processed those bytes


## 2. The fixture

Five candidates. The seal forbids call lineage `sibling-call` — the isolation
Part 5 depends on. The budget is 16.

In [3]:
print(f"{'id':<8}{'label':<8}{'kind':<12}{'size':<7}{'required':<11}{'lineage'}")
print("-" * 68)
for c in fixture["candidates"]:
    cid = c["candidate_id"]
    short = cid if len(cid) <= 6 else cid[:6] + "..."
    print(f"{short:<8}{c.get('label',''):<8}{c['kind']:<12}{c['size_tokens']:<7}"
          f"{str(c['required']):<11}{c.get('lineage_ids')}")

print()
print("Declared sizes are synthetic estimates, not tokenizer counts.")

id      label   kind        size   required   lineage
--------------------------------------------------------------------
A       A       event       4      True       []
8d8ce8...B       artifact    6      True       []
C       C       event       5      False      []
D       D       claim       3      False      ['source-call']
01ee15...E       artifact    7      False      ['sibling-call']

Declared sizes are synthetic estimates, not tokenizer counts.


## 3. What the run established

The bundle's own claim table, recomputed by an independent verifier that
imports no CodeAI code.

In [4]:
comparison = json.loads((SEL / "comparison.json").read_text(encoding="utf-8"))
claims = comparison["claims"]

print(f"{'claim':<34}{'verdict':<10}{'description'}")
print("-" * 104)
for c in claims:
    print(f"{c['claim_id']:<34}{c['verdict']:<10}{c['description'][:52]}")

verdicts = {c["verdict"] for c in claims}
print()
print("verdicts present:", verdicts)
assert verdicts == {"PASS"}
print("assertion held: every claim in the preserved comparison passed")

claim                             verdict   description
--------------------------------------------------------------------------------------------------------
reorder/0-identity                PASS      Canonical package and trace identities recompute
reorder/0-order                   PASS      Output IDs are canonically ordered within kind
reorder/0-trace                   PASS      Package selection agrees with trace and every exclus
H1-0                              PASS      Caller order does not change fixed-input selection o
H1-budget-0                       PASS      Declared estimate arithmetic and optional budget exc
reorder/1-identity                PASS      Canonical package and trace identities recompute
reorder/1-order                   PASS      Output IDs are canonically ordered within kind
reorder/1-trace                   PASS      Package selection agrees with trace and every exclus
H1-1                              PASS      Caller order does not change fixed-inpu

## 4. Arrival order does not change the selection

The same five candidates supplied in four different orders.

In [5]:
order_claims = [c for c in claims if c["claim_id"].startswith("reorder/")]
identities = {c["observed"] for c in order_claims if "identity" in c["claim_id"]}

for c in order_claims[:6]:
    print(f"  {c['claim_id']:<26}{c['verdict']:<8}{str(c['observed'])[:52]}")

print()
print("distinct package identities across the permutations:", len(identities))
print("   ", identities)
print()
print("Four permutations of one fixture are not a proof for all inputs. They do")
print("show that, for this input, nothing about arrival order leaked into what")
print("was selected or how it was identified.")

  reorder/0-identity        PASS    cfa87a17ed28097d90b37a316e5c12b0b2d090b8c541ad15e90c
  reorder/0-order           PASS    {'event_ids': ['A'], 'artifact_ids': ['8d8ce8070d3bb
  reorder/0-trace           PASS    ['01ee1583d81c73f3b3ee37ed419b3e8c6f063833b2759cfda1
  reorder/1-identity        PASS    cfa87a17ed28097d90b37a316e5c12b0b2d090b8c541ad15e90c
  reorder/1-order           PASS    {'event_ids': ['A'], 'artifact_ids': ['8d8ce8070d3bb
  reorder/1-trace           PASS    ['01ee1583d81c73f3b3ee37ed419b3e8c6f063833b2759cfda1

distinct package identities across the permutations: 1
    {'cfa87a17ed28097d90b37a316e5c12b0b2d090b8c541ad15e90ca2ee381571d6'}

Four permutations of one fixture are not a proof for all inputs. They do
show that, for this input, nothing about arrival order leaked into what
was selected or how it was identified.


## 5. Required material fails; it is not trimmed

Two failure cases, each of which produced **no package at all**.

In [6]:
for case in ("missing-required", "budget-required"):
    req = json.loads((SEL / case / "request.json").read_text(encoding="utf-8"))
    res = json.loads((SEL / case / "result.json").read_text(encoding="utf-8"))
    print(f"=== {case} ===")
    print("  budget    :", req.get("budget_tokens", req.get("budget")))
    print("  required  :", req.get("required_ids"))
    print("  outcome   :", json.dumps(res)[:200])
    print()

print("The compiler did not silently drop the source to make room, which would")
print("have left a model to produce an answer without ever seeing its source.")

=== missing-required ===
  budget    : 8
  required  : None
  outcome   : {"outcome": "failure", "exception": "RequiredContextMissing", "message": "REQUIRED_CONTEXT_MISSING: absent-required was required but not supplied", "package": null, "trace": null, "events_before": 5, 

=== budget-required ===
  budget    : None
  required  : None
  outcome   : {"outcome": "failure", "exception": "ContextBudgetUnsatisfiable", "message": "CONTEXT_BUDGET_UNSATISFIABLE: required 10 tokens exceed budget 9", "package": null, "trace": null}

The compiler did not silently drop the source to make room, which would
have left a model to produce an answer without ever seeing its source.


## 6. What the identity binds, and what it does not

In [7]:
ident = json.loads((SEL / "identity" / "payload.json").read_text(encoding="utf-8"))
print(json.dumps(ident, indent=1)[:900])

{
 "original": {
  "package_id": "d8394de3420d63f48b7083f8a4ecc3d1c354acee5624fc5ffbabed02bd613a7f",
  "task_id": "task-context",
  "actor": {
   "actor_id": "fixture-reader",
   "kind": "deterministic",
   "provider": null,
   "model": null,
   "version": "fixture-v1"
  },
  "prompt": "Review selected inputs.",
  "event_ids": [
   "A"
  ],
  "artifact_ids": [
   "8d8ce8070d3bb58d80f25b6f2d769e872cf82c0f93c2f7c89a9538f582392df1"
  ],
  "seal": {
   "forbidden_event_ids": [],
   "forbidden_call_ids": [
    "sibling-call"
   ],
   "forbidden_artifact_ids": [],
   "forbidden_lineage_ids": []
  },
  "metadata": {},
  "objective": "Select the context",
  "claim_ids": [
   "D"
  ],
  "budget_tokens": 8,
  "prompt_version": "context-fixture-v1",
  "trace_hash": "c543ab0d8e5f31743904a23c0860c0d8a222e2ad4d26396ded43938c7c2dad60",
  "provenance": {
   "trace_id": "c543ab0d8e5f31743904a23c0860c0d8a


The chapter's probe: the objective's text was changed from *"Review the
paragraph using its source."* to *"Changed under same ID"*, and **both
identities stayed exactly the same**.

> A package ID is a **selection identity**. It names which IDs were chosen
> under which task, actor, prompt, seal and budget. It is **not** a content
> hash of the material behind those IDs.

In [8]:
ghost = json.loads((SEL / "offered-missing-artifact.json").read_text(encoding="utf-8"))
pkg = ghost["package"]
print("a required artifact id with no bytes behind it:")
print("   artifact_ids :", pkg["artifact_ids"])
print("   package_id   :", pkg["package_id"][:24], "...")
print()
print("It was SELECTED, and the package was produced. Looking the id up in the")
print("artifact store fails.")
print()
print("Offering an id is not the same as proving its bytes exist, and the")
print("compiler checks only the first.")

a required artifact id with no bytes behind it:
   artifact_ids : ['ghost-artifact']
   package_id   : daaaaea12f2bebbbe51a8d72 ...

It was SELECTED, and the package was produced. Looking the id up in the
artifact store fails.

Offering an id is not the same as proving its bytes exist, and the
compiler checks only the first.


## 7. The gap: allowed to see is not what was sent

This is the chapter's finding. The experiment prepared, without sending, the
request that would carry the recorded package.

In [9]:
boundary = json.loads((SEL / "downstream-boundary.json").read_text(encoding="utf-8"))

print("prepared request body:")
print(json.dumps(boundary["body"], indent=1))
print()
print("selected ids  :", boundary.get("selected_ids", boundary.get("eligible_ids")))
print()
source_b = boundary["source_B_text"]
print("source paragraph B, whose text begins:", repr(source_b[:24]))

body_text = json.dumps(boundary["body"])
print()
print("does the prepared body contain B's text? ", source_b[:12] in body_text)
assert source_b[:12] not in body_text
print()
print("assertion held: the selected source paragraph is ABSENT from the request")

prepared request body:
{
 "model": "fixture",
 "messages": [
  {
   "role": "user",
   "content": "Review selected inputs."
  }
 ]
}

selected ids  : ['A', '8d8ce8070d3bb58d80f25b6f2d769e872cf82c0f93c2f7c89a9538f582392df1', 'C', 'D', '01ee1583d81c73f3b3ee37ed419b3e8c6f063833b2759cfda1ae279ef7a19f6c']

source paragraph B, whose text begins: 'SOURCE-B: preserved sour'

does the prepared body contain B's text?  False

assertion held: the selected source paragraph is ABSENT from the request


In [10]:
print("The same harness then read the SEALED artifact E straight from the")
print("artifact store, which worked:")
print("   ", repr(boundary["sealed_E_read_directly"][:48]))
print()
print("The seal had governed the PACKAGE, not the STORE.")
print()
print("So, in CodeAI as this run found it:")
print("  available / eligible / selected  -> decided, recorded, recoverable")
print("  rendered                         -> whatever the prompt string contains")
print("  received                         -> not observable at all")
print()
print("A context package is a record of PERMISSION. It becomes a record of")
print("INPUT only when the bytes sent can be traced back to it.")

The same harness then read the SEALED artifact E straight from the
artifact store, which worked:
    'SOURCE-E: sibling-derived content.'

The seal had governed the PACKAGE, not the STORE.

So, in CodeAI as this run found it:
  available / eligible / selected  -> decided, recorded, recoverable
  rendered                         -> whatever the prompt string contains
  received                         -> not observable at all

A context package is a record of PERMISSION. It becomes a record of
INPUT only when the bytes sent can be traced back to it.


## 8. How the gap was closed

Stage 15B made rendering **opt-in**: a call that sets
`context_render = "context-render-v1"` has its selected material resolved to
bytes, laid out canonically, and bound into the manifest before any effect.

In [11]:
if REN.exists():
    cases = sorted(p.name for p in REN.iterdir() if p.is_dir())
    print("rendered-run cases preserved:", cases)
    sem = REN / "verification-semantic.json"
    if sem.exists():
        v = json.loads(sem.read_text(encoding="utf-8"))
        claims_r = v.get("claims", [])
        passed = sum(1 for c in claims_r if c.get("verdict") == "PASS")
        print(f"\nindependent verifier claims: {passed}/{len(claims_r)} PASS")
        for c in claims_r[:6]:
            print(f"   {c.get('claim_id','')[:38]:<40}{c.get('verdict')}")
else:
    print("context-rendering bundle not present in this copy")

rendered-run cases preserved: ['before', 'failure-injection', 'ghost', 'legacy-control', 'mutation-claim', 'mutation-event', 'permutation', 'selected', 'tests']

independent verifier claims: 59/59 PASS
   selected:rerender                       PASS
   selected:rendered-sha                   PASS
   selected:items                          PASS
   selected:package-binding                PASS
   selected:one-send                       PASS
   selected:layout                         PASS


In [12]:
BINDING = {
    "package ID":        "a SELECTION identity (unchanged by a payload rewrite)",
    "per-item hash":     "names the source bytes of each rendered item",
    "rendered hash":     "names the composed context bytes, stored as an artifact",
    "request body hash": "names the prepared request carrying that text",
    "received":          "still not observable from this side of the API",
}
for k, v in BINDING.items():
    print(f"  {k:<20}{v}")

print()
print("The limits travel with the result, as the chapter states them:")
for lim in [
    "only calls that OPT IN are covered; other paths still send the prompt string",
    "the fan-out path was not covered by this stage (see Chapter 23)",
    "one adapter and one API protocol were exercised",
    "events and claims render as raw payload JSON, a provenance format",
    "seals are still NOT access control",
]:
    print("   -", lim)

  package ID          a SELECTION identity (unchanged by a payload rewrite)
  per-item hash       names the source bytes of each rendered item
  rendered hash       names the composed context bytes, stored as an artifact
  request body hash   names the prepared request carrying that text
  received            still not observable from this side of the API

The limits travel with the result, as the chapter states them:
   - only calls that OPT IN are covered; other paths still send the prompt string
   - the fan-out path was not covered by this stage (see Chapter 23)
   - one adapter and one API protocol were exercised
   - events and claims render as raw payload JSON, a provenance format
   - seals are still NOT access control


## Interpretation

Chapter 15's contribution is a distinction that most systems cannot make at
all, and its honesty is in having found its own gap:

1. **Context is an input, not a transcript.** Order candidates canonically,
   admit required material first, fail rather than trim, and record a reason
   for every exclusion.
2. **Know what an identity names.** A selection identity is not a content hash.
   The payload-rewrite probe is the cheapest possible demonstration.
3. **Tie the record to the request.** Until the bytes sent can be traced to the
   package, the package records permission and not input.
4. **Isolation is only as strong as the provenance it is given.** A seal
   excludes declared lineage. Remove the lineage and the same artifact is
   admitted — which is Chapter 23's whole subject.

## Try it yourself

1. **Re-derive a package identity.** Take the fixture, hash the task, actor,
   prompt, budget, seal and sorted selected ids, and compare with the recorded
   `package_id`. Then change one declared size and watch the identity *not*
   move.
2. **Add a content hash.** Extend the identity to cover the bytes behind each
   id. Which probe in section 6 now fails, and is that the behaviour you want?
3. **Write the ghost check.** Make the compiler refuse an id with no bytes in
   the store. What does that cost you when an artifact is legitimately produced
   later in the same run?
4. **Enumerate your own channels.** The chapter borrows Lampson's confinement
   problem: the seal blocks one channel. List every other route by which one
   call's output could reach another in your system — files, caches, tools,
   your own copy-paste.